# Train stacked BiLSTM and export raw ensemble predictions


## Setup


In [ ]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)



In [ ]:
%pip install -q -r "$repo_dir/requirements.txt"


In [ ]:
import gc
import shutil
from time import perf_counter
from importlib.metadata import version

import pandas as pd
import torch
from google.colab import drive
from tokenizers import Tokenizer

from src.data_io import get_data_dir, load_manifest, load_data_splits, save_json, file_sha256
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.training import set_seed, train_epoch, predict, save_checkpoint
from src.export_predictions import export_predictions

drive.mount("/content/drive")


## Configuration


In [ ]:
run_name = "bilstm_3l_h256_b128_s17_retrain"
config = {
    "model": "bilstm", "architecture": "stacked",
    "embedding_dim": 128, "hidden_size": 256, "num_layers": 3,
    "dropout": 0.2, "seed": 17, "batch_size": 128,
    "learning_rate": 1e-3, "epochs": 3, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk", "ru"], "optimizer": "Adam",
    "tokenizer": "bpe_bytelevel.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "validation_version": "v1",
    "metric_version": "kaggle_original_regex_v1",
}
decoding = "viterbi_lines"
run_dir = Path("/content/drive/MyDrive/UA-Locations-NER/stacked") / run_name
output_dir = Path("/content/drive/MyDrive/ner_results") / f"{run_name}_e{config['epochs']}_{decoding}"
if output_dir.exists():
    raise FileExistsError(f"Choose a new export folder: {output_dir}")
run_dir.mkdir(parents=True, exist_ok=False)
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]
checkpoint_path = run_dir / f"epoch_{config['epochs']:02d}.pt"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()

provenance = {
    "run_id": run_name, "seed": config["seed"],
    "git_revision": subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
    ).strip(),
    "git_status": subprocess.check_output(
        ["git", "-C", str(repo_dir), "status", "--short"], text=True,
    ).strip(),
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "notebook_sha256": file_sha256(repo_dir / "notebooks/06_export_predictions.ipynb"),
    "source_sha256": {path.name: file_sha256(path)
                      for path in sorted((repo_dir / "src").glob("*.py"))},
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
shutil.copy2(tokenizer_path, run_dir / tokenizer_path.name)
print(device, run_dir)


## Train, full fixed VAL and TEST


In [ ]:
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir)
expected_val_rows = len(manifest.loc[
    (manifest["split"] == "val") & manifest["source"].isin(config["sources"])
])
del manifest

train_records, val_records, alignment = [], [], []
for source in config["sources"]:
    source_splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,))
    for split in ("train", "val"):
        frame = source_splits.pop(f"{source}_{split}")[["text", "loc_markers"]]
        records, skipped = prepare_ner_records(
            tokenizer, frame, source, split, batch_size=config["prepare_batch_size"],
        )
        (train_records if split == "train" else val_records).extend(records)
        alignment.append({"source": source, "split": split, "rows": len(records),
                          "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_alignment_failures.json", skipped)
        del frame, records, skipped
        gc.collect()
    del source_splits

if len(val_records) != expected_val_rows:
    raise ValueError("Export requires all fixed validation rows")
pd.DataFrame(alignment).to_csv(run_dir / "alignment.csv", index=False)
display(pd.DataFrame(alignment))

test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
loader_options = {
    "pad_id": tokenizer.token_to_id("[PAD]"),
    "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"],
    "pin_memory": device.type == "cuda",
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)
test_loader = make_loader(test_records, **loader_options)
print("Full VAL:", len(val_records), "TEST:", len(test_records))


## New model


In [ ]:
set_seed(config["seed"])
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": tokenizer.token_to_id("[PAD]"),
    **{key: config[key] for key in ("embedding_dim", "hidden_size", "num_layers", "dropout")},
    "num_labels": len(LABEL_TO_ID),
}
model = BiLSTMTagger(**model_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
save_json(run_dir / "model_config.json", model_config)
print(model)


## Train and save each epoch to Drive


In [ ]:
history, best_score = [], -1
for epoch in range(1, config["epochs"] + 1):
    started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device)
    scores = competition_scores(val_records, predictions)
    del predictions
    result = {
        "epoch": epoch, "train_loss": train_loss,
        "epoch_seconds": perf_counter() - started,
        **{f"{source}_val_loss": value for source, value in val_losses.items()}, **scores,
    }
    history.append(result)
    save_checkpoint(run_dir / f"epoch_{epoch:02d}.pt", model, optimizer, epoch,
                    config, model_config, scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score = scores[config["selection_metric"]]
        save_checkpoint(run_dir / "best.pt", model, optimizer, epoch,
                        config, model_config, scores, provenance)
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    print(result)

display(pd.DataFrame(history))
optimizer.zero_grad(set_to_none=True)
del optimizer
gc.collect()


## Export selected epoch: raw viterbi_lines spans


In [ ]:
def predict_for_export(model, loader):
    return predict(model, loader, device, decoding=decoding)

export_predictions(
    output_dir=output_dir, checkpoint_path=checkpoint_path,
    model=model, predict_fn=predict_for_export,
    val_records=val_records, val_loader=val_loader,
    test_records=test_records, test_loader=test_loader,
)

selected = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
metadata = {
    "checkpoint": checkpoint_path.name, "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": file_sha256(checkpoint_path), "epoch": selected["epoch"],
    "model_config": selected["model_config"], "training_config": selected["config"],
    "checkpoint_provenance": selected["provenance"],
    "decoding": decoding, "postprocessing": "none", "chunking": False,
    "tokenizer": tokenizer_path.name, "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "val_rows": len(val_records), "test_rows": len(test_records),
    "span_coordinates": "Python Unicode character offsets [start, end) in original text",
    "source_sha256": {path.name: file_sha256(path)
                      for path in sorted((repo_dir / "src").glob("*.py"))},
}
del selected
save_json(output_dir / "metadata.json", metadata)
print(output_dir)
display(pd.DataFrame([
    {"file": path.name, "size_MB": round(path.stat().st_size / 1024**2, 2)}
    for path in sorted(output_dir.iterdir()) if path.is_file()
]))
